## Script for Comparing Incorrectly Labeled Epochs

In [96]:
import numpy as np 
import pandas as pd
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
from matplotlib import pyplot as plt
import tkinter as tk
import glob
import pickle
from sklearn.model_selection import train_test_split, KFold
import matplotlib
matplotlib.use('QtAgg') 

In [ ]:
# importing model 
with open("M2_all.pkl", "rb") as f:
    model = pickle.load(f)

In [ ]:
# importing features dataframe 
features_all = pd.read_pickle("training_features_19032026.pkl")

In [ ]:
X_zygo = features_all[["Zygo"]] 
X_corr = features_all[["Corr"]]

X_zygo = X_zygo.to_numpy()
X_zygo = np.array(X_zygo.tolist())
X_zygo = np.transpose(X_zygo, (0, 2, 1)) # setting dimensions 

X_corr = X_corr.to_numpy()
X_corr = np.array(X_corr.tolist())
X_corr = np.transpose(X_corr, (0, 2, 1)) # setting dimensions 

y_zygo = features_all["Num_Contractions_Zygo"].astype(int).to_numpy()
y_corr = features_all["Num_Contractions_Corr"].astype(int).to_numpy()

X = np.concatenate((X_zygo, X_corr), axis=0)
y = np.concatenate((y_zygo,y_corr),axis=0)       


indices = np.arange(len(X))
idx_train, idx_test = train_test_split(indices, test_size=0.2, random_state=42) # keep 20% purely for testing 

In [ ]:
print(np.any(np.concatenate([y_pred_test_zygo, y_pred_test_corr]) != y_pred_test))

In [ ]:
n1 = len(X_zygo)
labels = np.where(idx_test < n1, "Zygo", "Corr")

idx_test_1 = idx_test[idx_test < n1]
idx_test_2 = idx_test[idx_test >= n1] 
X_test_zygo = X[idx_test_1]
X_test_corr = X[idx_test_2]
X_test = X[idx_test]

# get testing results 
y_pred_test_zygo = model.predict(X_test_zygo)   
y_pred_test_corr = model.predict(X_test_corr)   
y_pred_test  = model.predict(X_test)

y_pred_test_zygo = np.argmax(y_pred_test_zygo, axis=1)   
y_pred_test_corr = np.argmax(y_pred_test_corr, axis=1)   
y_pred_test = np.argmax(y_pred_test,axis=1)

# get other fields for dataframe 
row_idx = np.where(idx_test < n1, idx_test, idx_test - n1)

subjects = features_all["Subject"].to_numpy().squeeze()
subject_subset = subjects[row_idx]

epochs = features_all["Triggers_Order_Nap"].to_numpy()
epochs_subset = epochs[row_idx]

naps = features_all["Nap Number"].to_numpy()
naps_subset = naps[row_idx]
# get mismatched epoch indices where the tested indices dont equal predicted 
epoch_idx = np.where(y[idx_test] != y_pred_test)[0]

mismatched_subj = subject_subset[epoch_idx]
mismatched_epochs = epochs_subset[epoch_idx]
mismatched_naps = naps_subset[epoch_idx]
mismatched_labels = labels[epoch_idx]

Pre-Processing

In [ ]:
# making dataframe for epoch rescoring 
rescore_epochs = pd.DataFrame({
    "Subject": mismatched_subj,
    "Nap Number": mismatched_naps,
    "Triggers_Order_Nap":mismatched_epochs,
    "Label": labels[epoch_idx],
    "Prediction": y_pred_test[epoch_idx]
}, index=epoch_idx)

rescore_epochs.to_excel("rescore_epochs.xlsx", index=False)
rescore_epochs_reshape = (
    rescore_epochs
    .pivot_table(
        index=["Subject", "Nap Number", "Triggers_Order_Nap"],
        columns="Label",
        values="Prediction",
        aggfunc="first"
    )
    .rename(columns={
        "Zygo": "Prediction_Zygo",
        "Corr": "Prediction_Corr"
    })
    .reset_index()
)

In [ ]:
# making final dataframe for going through bad epochs 
rows = []
seen = set()

for _, row in rescore_epochs_reshape.iterrows():
    key = (row["Subject"], row["Nap Number"], row["Triggers_Order_Nap"])

    # skip duplicates
    if key in seen:
        continue
    seen.add(key)

    # find matching row(s) in the other dataframe
    match = features_all[
        (features_all["Subject"] == row["Subject"]) &
        (features_all["Nap Number"] == row["Nap Number"]) &
        (features_all["Triggers_Order_Nap"] == row["Triggers_Order_Nap"])
    ]

    # if there is a match, take the first one
    if not match.empty:
        match_row = match.iloc[0]

        match_zygo = match_row["Zygo"]
        match_corr = match_row["Corr"]

        pred_zygo = row["Prediction_Zygo"]
        pred_corr = row["Prediction_Corr"]

        if pd.isna(pred_zygo):
            pred_zygo = np.argmax(model.predict(np.array(match_zygo.tolist()).reshape(1, 2251,1), verbose=0))

        if pd.isna(pred_corr):
            pred_corr = np.argmax(model.predict(np.array(match_corr.tolist()).reshape(1, 2251,1), verbose=0))


        rows.append({
            "Subject": row["Subject"],
            "Nap Number": row["Nap Number"],
            "Triggers_Order_Nap": row["Triggers_Order_Nap"],
            "Prediction_Zygo": pred_zygo,
            "Prediction_Corr": pred_corr,

            # examples of fields from df_other
            "Num_Contractions_Zygo": match_row["Num_Contractions_Zygo"],
            "Num_Contractions_Corr": match_row["Num_Contractions_Corr"],
            "Zygo": match_row["Zygo"],
            "Corr": match_row["Corr"],
        })

    else:
        # if no match exists, still keep the row
        rows.append({
            "Subject": row["Subject"],
            "Nap Number": row["Nap Number"],
            "Triggers_Order_Nap": row["Triggers_Order_Nap"],
            "Label": row["Label"],
            "Prediction_Zygo": pred_zygo,
            "Prediction_Corr": pred_corr,

            "Num_Contractions_Zygo": pd.NA,
            "Num_Contractions_Corr": pd.NA,
            "Zygo": pd.NA,
            "Corr": pd.NA
        })


mismatch_df = pd.DataFrame(rows)    


In [ ]:
#mismatch_df.insert(0, "Epoch", np.arange(np.shape(mismatch_df)[0])+1)
mismatch_df.head(10)
mismatch_df.to_pickle("mismatch_df.pkl")
#mismatch_df.to_excel("mismatch_df.xlsx", index=False)

In [ ]:
mismatch_df.head()

Scoring for Mismatched Epochs

In [ ]:
# load in subset of epochs 
mismatch_df = pd.read_pickle("mismatch_df.pkl")


In [ ]:
# Load in entire first state 
def load_saved_results_from_df(df):
    saved = {}

    for _, row in df.iterrows():
        key = (row["Epoch"], row["Muscle_type"])

        saved[key] = {
            "Epoch": row["Epoch"],
            "Muscle_type": row["Muscle_type"],
            "Response_start_sample": row["Response_start_sample"],
            "Response_end_sample": row["Response_end_sample"],
            "Contraction_number": row["Contraction_number"],
        }

    return saved


In [ ]:
current_index = 0 
loading = True # SET TO TRUE IF LOADING IN DATFRAME 

# data frame containing trigger infromation for epochs 
df_triggers = pd.DataFrame(columns=[
    "Epoch",
    "Muscle_type",
    "Response_start_sample",
    "Response_end_sample",
    "Contraction_number"
])

if loading: 
    df_triggers = pd.read_excel("zeynep_rescore.xlsx") # REPLACE WITH EXCEL FILE YOU WANT TO LOAD IN 


# tracking current state of scoring 
state = {
    "current_index": 0,          
    "selected_muscle": None,      
    "reset_mode": False,
    "fig": None,
    "axes": None,
    "saved_results": {},         
}


# scoring starts with no epochs 
def blank_epoch_state():
    return {
        "Corr": {"start": None, "end": None, "count": None},
        "Zygo": {"start": None, "end": None, "count": None},
    }


state["current_state"] = blank_epoch_state()

state["saved_results"] = load_saved_results_from_df(df_triggers)

In [ ]:
all_states_array = []
state["current_state"]
 

In [91]:
# GUI
frq = 250


# makes sure all elementsof scoring are complete 
def scoring_complete(muscle):
    
    s = state["current_state"][muscle]
    count = s["count"]
    start = s["start"]
    end = s["end"]


    if count is None:
        return False
    if count == 0:
        return True
    return start is not None and end is not None


# checks if scoring has started for muscle 
def scoring_started(muscle):
    s = state["current_state"][muscle]
    return any(v is not None for v in [s["start"], s["end"], s["count"]])


# rebuilds df based on current scores 
def update_df():
    global df_triggers

    if len(state["saved_results"]) == 0:
        df_triggers = pd.DataFrame(columns=[
            "Epoch",
            "Muscle_type",
            "Response_start_sample",
            "Response_end_sample",
            "Contraction_number"
        ])
        return

    rows = list(state["saved_results"].values())
    df_triggers = (
        pd.DataFrame(rows)
        .sort_values(["Epoch", "Muscle_type"])
        .reset_index(drop=True)
    )


def sync_one_muscle_to_saved(muscle):
    current_index = state["current_index"]
    s = state["current_state"][muscle]

    if scoring_complete(muscle):
        state["saved_results"][(current_index, muscle)] = {
            "Epoch": current_index,
            "Muscle_type": muscle,
            "Response_start_sample": s["start"],
            "Response_end_sample": s["end"],
            "Contraction_number": s["count"],
        }
    else:
        state["saved_results"].pop((current_index, muscle), None)

    update_df()


def sync_current_epoch_to_saved():
    # updates data frame for each muscle 
    sync_one_muscle_to_saved("Corr")
    sync_one_muscle_to_saved("Zygo")


# loads current state of epoch 
def load_current_epoch_state():
    global current_index, state

    
    current = blank_epoch_state()
    

    for muscle in ["Corr", "Zygo"]:
        key = (current_index, muscle)
        if key in state["saved_results"]:
            row = state["saved_results"][key]

            current[muscle]["start"] = row["Response_start_sample"] 
            current[muscle]["end"] = row["Response_end_sample"] 
            current[muscle]["count"] = row["Contraction_number"] 
 
    
  
    state["current_state"] = current
    state["selected_muscle"] = None
    state["reset_mode"] = False


# function for resetting muscle if press "r"
def reset_muscle(muscle):
    global current_index 

    state["current_state"][muscle] = {"start": None, "end": None, "count": None}
    state["saved_results"].pop((current_index, muscle), None)
    update_df()


# call when move to next epoch 
def move_epoch(step):
    global state, current_index
   
    # save anything complete before leaving current epoch
    sync_current_epoch_to_saved()
    n = len(mismatch_df)

    state["current_index"] = (state["current_index"] + step) % n
    current_index = state["current_index"] 
    load_current_epoch_state()

    # print("CURRENT DATAFRAME")
    # print(df_triggers)
    all_states_array.append(state["current_state"])
    print(all_states_array)
    
    plt.close()
    plot_figure(state["current_index"])


# generates title for muscle based on state of scoring 
def title_for_muscle(muscle):
    global state
    
    s = state["current_state"][muscle]
    count = s["count"]
    start = s["start"]
    end = s["end"]



    if scoring_complete(muscle):
        if count == 0:
            return f"{muscle}: SCORED | CONTRACTIONS=0", "green"
        return f"{muscle}: SCORED | CONTRACTIONS={count}, START={start}, END={end}", "green"

    if scoring_started(muscle):
        return f"{muscle}: UNFINISHED SCORING | CONTRACTIONS={count}, START={start}, END={end}", "black"

    return f"{muscle}: NOT SCORED","red"



def plot_figure(t):
    global state


    fig, ax = plt.subplots(2, 1, sharex=False)
    plt.subplots_adjust(hspace=0.5) 

    state["fig"] = fig
    state["axes"] = ax

    zygo = mismatch_df["Zygo"].iloc[t] 
    corr = mismatch_df["Corr"].iloc[t] 

    ymax_emg = max(np.max(np.array(list(zygo))),np.max(np.array(list(corr)))) 
    

    title_zygo,color_zygo = title_for_muscle("Zygo")
    title_corr,color_corru = title_for_muscle("Corr")

    fig.suptitle(f"Epoch {t+1}")

    # Corr subplot
    ax[0].plot(corr, color="blue", label="Corr")

    ax[0].set_ylim(-200, 200)

    ax[0].set_ylabel("Corr EMG [V]")
    ax[0].set_xlabel("Samples")
    ax[0].set_title(title_corr,color=color_corru)

    # Zygo subplot
    ax[1].plot(zygo, label="Zygo",color="black")
    ax[1].set_ylim(-200, 200)

    ax[1].set_ylabel("Zygo EMG [V]")
    ax[1].set_xlabel("Samples")
    ax[1].set_title(title_zygo,color=color_zygo)    


    fig.canvas.mpl_connect("button_press_event", on_click)
    fig.canvas.mpl_connect("key_press_event", on_key)


    plt.show()


# EVENTS
def on_click(event):
    global current_index 
    if event.inaxes is None or event.xdata is None:
        return

    ax_corr, ax_zygo = state["axes"]

    if event.inaxes == ax_corr:
        muscle = "Corr"
    elif event.inaxes == ax_zygo:
        muscle = "Zygo"
    else:
        return

    x = int(round(event.xdata))
    s = state["current_state"][muscle]
    state["selected_muscle"] = muscle

    if s["start"] is None:
        s["start"] = x
    elif s["end"] is None:
        s["end"] = x
    else:
        s["start"] = x
        s["end"] = None

    plt.close()
    plot_figure(current_index)


def on_key(event):
    global current_index
    key = event.key

    

    if key is None:
        return

    allowed_keys = {'0','1', '2', '3', '4', '5','6', '7', '8', '9'}

    # score selected muscle
    if key in allowed_keys:
        muscle = state["selected_muscle"]
        if muscle is None:
            print("Select a muscle first by clicking on its subplot.")
            return

        state["current_state"][muscle]["count"] = int(key)

        # immediately save completed state
        sync_one_muscle_to_saved(muscle)
        plt.close()
        plot_figure(current_index)

    # navigation
    elif key == "right":
        #current_index = (current_index + 1) % len(mismatch_df)  #
        move_epoch(1)

    elif key == "left":
        #current_index = (current_index - 1) % len(mismatch_df)  #
        move_epoch(-1)

    # reset mode
    elif key == "r":
        state["reset_mode"] = True
        plt.close()
        plot_figure(current_index)
 
    # reset Corr
    elif key == "c":
        if state["reset_mode"] or True:
            reset_muscle("Corr")
            state["selected_muscle"] = "Corr"
            state["reset_mode"] = False
            plt.close()
            plot_figure(current_index)
 
    # reset Zygo
    elif key == "z":
        if state["reset_mode"] or True:
            reset_muscle("Zygo")
            state["selected_muscle"] = "Zygo"
            state["reset_mode"] = False
            plt.close()
            plot_figure(current_index)
 
    # quit
    elif key in ["q", "escape"]:
        print("Quitting scorer.")
        plt.close(state["fig"])
        df_triggers.to_excel("zeynep_rescore.xlsx", index=False) # saving current state if crashes
        
 


load_current_epoch_state()
plot_figure(current_index)

[{'Corr': {'start': 157, 'end': 1943, 'count': 0}, 'Zygo': {'start': 222, 'end': 736, 'count': 3}}, {'Corr': {'start': 12, 'end': 2233, 'count': 0}, 'Zygo': {'start': 1449, 'end': 1739, 'count': 1}}, {'Corr': {'start': 47, 'end': 2188, 'count': 0}, 'Zygo': {'start': 1519, 'end': 2113, 'count': 2}}, {'Corr': {'start': 661, 'end': 1110, 'count': 3}, 'Zygo': {'start': 421, 'end': 651, 'count': 2}}, {'Corr': {'start': 17, 'end': 2213, 'count': 0}, 'Zygo': {'start': 97, 'end': 2123, 'count': 0}}, {'Corr': {'start': 322, 'end': 651, 'count': 3}, 'Zygo': {'start': 7, 'end': 2223, 'count': 0}}, {'Corr': {'start': 307, 'end': 741, 'count': 4}, 'Zygo': {'start': -8, 'end': 2218, 'count': 0}}, {'Corr': {'start': 1040, 'end': 1389, 'count': 1}, 'Zygo': {'start': 27, 'end': 2218, 'count': 0}}, {'Corr': {'start': 431, 'end': 726, 'count': 3}, 'Zygo': {'start': 177, 'end': 382, 'count': 1}}, {'Corr': {'start': 217, 'end': 611, 'count': 3}, 'Zygo': {'start': 22, 'end': 2158, 'count': 0}}, {'Corr': {'s

QCoreApplication::exec: The event loop is already running


[{'Corr': {'start': 157, 'end': 1943, 'count': 0}, 'Zygo': {'start': 222, 'end': 736, 'count': 3}}, {'Corr': {'start': 12, 'end': 2233, 'count': 0}, 'Zygo': {'start': 1449, 'end': 1739, 'count': 1}}, {'Corr': {'start': 47, 'end': 2188, 'count': 0}, 'Zygo': {'start': 1519, 'end': 2113, 'count': 2}}, {'Corr': {'start': 661, 'end': 1110, 'count': 3}, 'Zygo': {'start': 421, 'end': 651, 'count': 2}}, {'Corr': {'start': 17, 'end': 2213, 'count': 0}, 'Zygo': {'start': 97, 'end': 2123, 'count': 0}}, {'Corr': {'start': 322, 'end': 651, 'count': 3}, 'Zygo': {'start': 7, 'end': 2223, 'count': 0}}, {'Corr': {'start': 307, 'end': 741, 'count': 4}, 'Zygo': {'start': -8, 'end': 2218, 'count': 0}}, {'Corr': {'start': 1040, 'end': 1389, 'count': 1}, 'Zygo': {'start': 27, 'end': 2218, 'count': 0}}, {'Corr': {'start': 431, 'end': 726, 'count': 3}, 'Zygo': {'start': 177, 'end': 382, 'count': 1}}, {'Corr': {'start': 217, 'end': 611, 'count': 3}, 'Zygo': {'start': 22, 'end': 2158, 'count': 0}}, {'Corr': {'s

QCoreApplication::exec: The event loop is already running
QCoreApplication::exec: The event loop is already running


[{'Corr': {'start': 157, 'end': 1943, 'count': 0}, 'Zygo': {'start': 222, 'end': 736, 'count': 3}}, {'Corr': {'start': 12, 'end': 2233, 'count': 0}, 'Zygo': {'start': 1449, 'end': 1739, 'count': 1}}, {'Corr': {'start': 47, 'end': 2188, 'count': 0}, 'Zygo': {'start': 1519, 'end': 2113, 'count': 2}}, {'Corr': {'start': 661, 'end': 1110, 'count': 3}, 'Zygo': {'start': 421, 'end': 651, 'count': 2}}, {'Corr': {'start': 17, 'end': 2213, 'count': 0}, 'Zygo': {'start': 97, 'end': 2123, 'count': 0}}, {'Corr': {'start': 322, 'end': 651, 'count': 3}, 'Zygo': {'start': 7, 'end': 2223, 'count': 0}}, {'Corr': {'start': 307, 'end': 741, 'count': 4}, 'Zygo': {'start': -8, 'end': 2218, 'count': 0}}, {'Corr': {'start': 1040, 'end': 1389, 'count': 1}, 'Zygo': {'start': 27, 'end': 2218, 'count': 0}}, {'Corr': {'start': 431, 'end': 726, 'count': 3}, 'Zygo': {'start': 177, 'end': 382, 'count': 1}}, {'Corr': {'start': 217, 'end': 611, 'count': 3}, 'Zygo': {'start': 22, 'end': 2158, 'count': 0}}, {'Corr': {'s

...

[{'Corr': {'start': 157, 'end': 1943, 'count': 0}, 'Zygo': {'start': 222, 'end': 736, 'count': 3}}, {'Corr': {'start': 12, 'end': 2233, 'count': 0}, 'Zygo': {'start': 1449, 'end': 1739, 'count': 1}}, {'Corr': {'start': 47, 'end': 2188, 'count': 0}, 'Zygo': {'start': 1519, 'end': 2113, 'count': 2}}, {'Corr': {'start': 661, 'end': 1110, 'count': 3}, 'Zygo': {'start': 421, 'end': 651, 'count': 2}}, {'Corr': {'start': 17, 'end': 2213, 'count': 0}, 'Zygo': {'start': 97, 'end': 2123, 'count': 0}}, {'Corr': {'start': 322, 'end': 651, 'count': 3}, 'Zygo': {'start': 7, 'end': 2223, 'count': 0}}, {'Corr': {'start': 307, 'end': 741, 'count': 4}, 'Zygo': {'start': -8, 'end': 2218, 'count': 0}}, {'Corr': {'start': 1040, 'end': 1389, 'count': 1}, 'Zygo': {'start': 27, 'end': 2218, 'count': 0}}, {'Corr': {'start': 431, 'end': 726, 'count': 3}, 'Zygo': {'start': 177, 'end': 382, 'count': 1}}, {'Corr': {'start': 217, 'end': 611, 'count': 3}, 'Zygo': {'start': 22, 'end': 2158, 'count': 0}}, {'Corr': {'s

QCoreApplication::exec: The event loop is already running


[{'Corr': {'start': 157, 'end': 1943, 'count': 0}, 'Zygo': {'start': 222, 'end': 736, 'count': 3}}, {'Corr': {'start': 12, 'end': 2233, 'count': 0}, 'Zygo': {'start': 1449, 'end': 1739, 'count': 1}}, {'Corr': {'start': 47, 'end': 2188, 'count': 0}, 'Zygo': {'start': 1519, 'end': 2113, 'count': 2}}, {'Corr': {'start': 661, 'end': 1110, 'count': 3}, 'Zygo': {'start': 421, 'end': 651, 'count': 2}}, {'Corr': {'start': 17, 'end': 2213, 'count': 0}, 'Zygo': {'start': 97, 'end': 2123, 'count': 0}}, {'Corr': {'start': 322, 'end': 651, 'count': 3}, 'Zygo': {'start': 7, 'end': 2223, 'count': 0}}, {'Corr': {'start': 307, 'end': 741, 'count': 4}, 'Zygo': {'start': -8, 'end': 2218, 'count': 0}}, {'Corr': {'start': 1040, 'end': 1389, 'count': 1}, 'Zygo': {'start': 27, 'end': 2218, 'count': 0}}, {'Corr': {'start': 431, 'end': 726, 'count': 3}, 'Zygo': {'start': 177, 'end': 382, 'count': 1}}, {'Corr': {'start': 217, 'end': 611, 'count': 3}, 'Zygo': {'start': 22, 'end': 2158, 'count': 0}}, {'Corr': {'s

QCoreApplication::exec: The event loop is already running


KeyboardInterrupt: 

In [83]:
np.shape(all_states_array[:220] ) 

(220,)

In [92]:
df_final = pd.DataFrame(columns=[
    "Epoch",
    "Muscle_type",
    "Response_start_sample",
    "Response_end_sample",
    "Contraction_number"
], index=range(438))

epoch_ct = 1
df_ct = 0 
 

all_states_array_real = all_states_array[:218] 

for state_temp in all_states_array_real:
    row_c = {"Epoch": epoch_ct, 
    "Muscle_type": "Corr", 
    "Response_start_sample": state_temp["Corr"]["start"],
    "Response_end_sample": state_temp["Corr"]["end"],
    "Contraction_number": state_temp["Corr"]["count"]
    }

    row_z = {"Epoch": epoch_ct, 
    "Muscle_type": "Zygo", 
    "Response_start_sample": state_temp["Zygo"]["start"],
    "Response_end_sample": state_temp["Zygo"]["end"],
    "Contraction_number": state_temp["Zygo"]["count"]
    }
    
    df_final.iloc[df_ct] = row_c
    df_final.iloc[df_ct+1] = row_z


    epoch_ct+=1
    df_ct+= 2

 

In [95]:
df_final.head(30)

,Epoch,Muscle_type,Response_start_sample,Response_end_sample,Contraction_number
0,1,Corr,157,1943,0
1,1,Zygo,222,736,3
2,2,Corr,12,2233,0
3,2,Zygo,1449,1739,1
4,3,Corr,47,2188,0
5,3,Zygo,1519,2113,2
6,4,Corr,661,1110,3
7,4,Zygo,421,651,2
8,5,Corr,17,2213,0
9,5,Zygo,97,2123,0


In [93]:
df_final.to_excel("final_scores_16042026_v3.xlsx", index=False)

In [94]:
df_triggers.head(30)

,Epoch,Muscle_type,Response_start_sample,Response_end_sample,Contraction_number
0,0,Corr,157,1943,0
1,0,Zygo,222,736,3
2,1,Corr,12,2233,0
3,1,Zygo,1449,1739,1
4,2,Corr,47,2188,0
5,2,Zygo,1519,2113,2
6,3,Corr,661,1110,3
7,3,Zygo,421,651,2
8,4,Corr,17,2213,0
9,4,Zygo,97,2123,0


In [ ]:
df_triggers.to_excel("final_scores_16042026.xlsx", index=False)